# Bias Tuning — BitFit

A linear transformation can be written as:

y = Wx + b

W: weight matrix
b: bias vector

In this exercise:
- Freeze all parameters.
- Enable training only for bias parameters.
- Train on manufacturing maintenance text.
- Compare domain-text loss before and after training.
- Verify that the frozen parameters did not change.

"Bias" here means a numerical model parameter, not social bias.

This is a small teaching experiment, not a production domain model.

In [7]:
%pip install -q torch "transformers>=4.45,<5" pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import hashlib
import math
import random

import pandas as pd
import torch

from IPython.display import display
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

# Choose: "cpu", "cuda", or "auto"
DEVICE_CHOICE = "cpu"

if DEVICE_CHOICE == "auto":
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
elif DEVICE_CHOICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError(
            'CUDA is unavailable. Set DEVICE_CHOICE = "cpu".'
        )
    device = torch.device("cuda")
elif DEVICE_CHOICE == "cpu":
    device = torch.device("cpu")
else:
    raise ValueError('Choose "cpu", "cuda", or "auto".')

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [9]:
MODEL_NAME = "distilbert/distilgpt2"

MAX_LENGTH = 96
BATCH_SIZE = 1
TRAINING_STEPS = 30
LEARNING_RATE = 5e-4

print("Model:", MODEL_NAME)
print("Training steps:", TRAINING_STEPS)

Model: distilbert/distilgpt2
Training steps: 30


# Create separate training and evaluation texts

In [10]:
train_texts = [
    "Predictive maintenance uses equipment condition data to plan "
    "maintenance before a failure occurs.",

    "Preventive maintenance follows a planned schedule based on "
    "time or equipment usage.",

    "Vibration monitoring helps engineers identify changes in "
    "the condition of rotating machinery.",

    "Bearing faults can produce characteristic vibration patterns. "
    "Engineers examine frequency information and operating conditions.",

    "An increase in bearing temperature may indicate lubrication "
    "problems, increased load, or other developing issues.",

    "Root mean square vibration summarizes the magnitude of a "
    "vibration signal over a measurement interval.",

    "A condition monitoring baseline describes normal equipment "
    "behavior under known operating conditions.",

    "Sensor trends should be compared with operating speed and load "
    "because these factors can influence measurements.",

    "A single abnormal reading does not confirm a machine fault. "
    "Additional measurements help support a diagnosis.",

    "Lubrication reduces friction and wear between moving surfaces. "
    "Incorrect lubrication can contribute to bearing damage.",

    "Shaft misalignment can increase vibration and mechanical stress "
    "in rotating equipment.",

    "Maintenance decisions should combine sensor measurements, "
    "inspection findings, and equipment history.",
]

eval_texts = [
    "Condition-based maintenance uses observations of machine health "
    "to decide when maintenance is needed.",

    "A technician investigating elevated vibration should consider "
    "changes in load, speed, alignment, and bearing condition.",

    "Rising bearing temperature is a symptom that requires further "
    "investigation rather than an immediate fault diagnosis.",

    "Comparing measurements with a normal operating baseline helps "
    "identify changes in equipment condition.",
]

print("Training texts:", len(train_texts))
print("Evaluation texts:", len(eval_texts))

assert set(train_texts).isdisjoint(eval_texts)

Training texts: 12
Evaluation texts: 4


# Load the tokenizer and model

In [11]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# tokenizer.pad_token = tokenizer.eos_token

# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
# model.config.pad_token_id = tokenizer.pad_token_id
# model.config.use_cache = False

# model = model.to(device)

# print("Model loaded.")

MODEL_NAME = "./workshop_models/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    attn_implementation="eager",
).to(device)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

# Freeze weights and enable bias training

In [12]:
for name, parameter in model.named_parameters():
    parameter.requires_grad = name.endswith(".bias")

trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

total_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in trainable_parameters)

assert trainable_count > 0, "No bias parameters were found."

print(f"Total parameters:     {total_count:,}")
print(f"Trainable parameters: {trainable_count:,}")
print(f"Trainable percentage: {100 * trainable_count / total_count:.4f}%")

parameter_table = pd.DataFrame([
    {
        "parameter": name,
        "shape": str(tuple(parameter.shape)),
        "elements": parameter.numel(),
    }
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
])

display(parameter_table)

Total parameters:     81,912,576
Trainable parameters: 51,456
Trainable percentage: 0.0628%


,parameter,shape,elements
0,transformer.h.0.ln_1.bias,"(768,)",768
1,transformer.h.0.attn.c_attn.bias,"(2304,)",2304
2,transformer.h.0.attn.c_proj.bias,"(768,)",768
3,transformer.h.0.ln_2.bias,"(768,)",768
4,transformer.h.0.mlp.c_fc.bias,"(3072,)",3072
5,transformer.h.0.mlp.c_proj.bias,"(768,)",768
6,transformer.h.1.ln_1.bias,"(768,)",768
7,transformer.h.1.attn.c_attn.bias,"(2304,)",2304
8,transformer.h.1.attn.c_proj.bias,"(768,)",768
9,transformer.h.1.ln_2.bias,"(768,)",768


### requires_grad=True allows gradients to be computed for those parameters. We will also give the optimizer only those parameters.

In [13]:
# Prepare batches
#The model learns to predict the next token. Padding positions are excluded from the loss using -100.

SyntaxError: invalid syntax (76633035.py, line 2)

In [15]:
def collate_texts(texts):
    batch = tokenizer(
        [text + tokenizer.eos_token for text in texts],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels

    return batch


train_loader = DataLoader(
    train_texts,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_texts,
)

eval_loader = DataLoader(
    eval_texts,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_texts,
)

print("Data loaders ready.")

Data loaders ready.


In [16]:
# Defining  evaluation and generation functions
#Evaluation averages loss over valid predicted tokens.

SyntaxError: invalid syntax (3259722538.py, line 2)

In [18]:
@torch.no_grad()
def evaluate_loss(loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        output = model(**batch)

        # Causal language models predict labels from position 1 onward.
        valid_tokens = (
            batch["labels"][:, 1:] != -100
        ).sum().item()

        total_loss += output.loss.item() * valid_tokens
        total_tokens += valid_tokens

    if total_tokens == 0:
        raise ValueError("No evaluation tokens were found.")

    return total_loss / total_tokens


@torch.no_grad()
def complete_text(prompt):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=45,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    prompt_length = inputs["input_ids"].shape[1]

    return tokenizer.decode(
        output_ids[0, prompt_length:],
        skip_special_tokens=True,
    ).strip()

In [19]:
# Recording the before-training baseline

In [20]:
def parameter_fingerprints():
    return {
        name: hashlib.sha256(
            parameter.detach().cpu().contiguous().numpy().tobytes()
        ).hexdigest()
        for name, parameter in model.named_parameters()
    }


before_fingerprints = parameter_fingerprints()
before_eval_loss = evaluate_loss(eval_loader)

test_prompts = [
    "Predictive maintenance is",
    "When bearing temperature increases,",
    "Vibration monitoring helps engineers",
]

before_completions = {
    prompt: complete_text(prompt)
    for prompt in test_prompts
}

print(f"Before-training evaluation loss: {before_eval_loss:.4f}")
print(f"Before-training perplexity: {math.exp(before_eval_loss):.2f}")

for prompt, completion in before_completions.items():
    print("\nPROMPT:", prompt)
    print("CONTINUATION:", completion)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Before-training evaluation loss: 5.4397
Before-training perplexity: 230.38

PROMPT: Predictive maintenance is
CONTINUATION: a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is

PROMPT: When bearing temperature increases,
CONTINUATION: the temperature increases, and the temperature decreases.

PROMPT: Vibration monitoring helps engineers
CONTINUATION: to understand the impact of the stress on the brain.


In [21]:
#Loss: lower means the model assigns higher probability to the evaluation text.
#Perplexity: another expression of the same loss; lower is better on the same evaluation text and tokenizer.

# Train only the biases

In [22]:
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.0,
)

training_history = []
train_iterator = iter(train_loader)

model.train()

for step in range(1, TRAINING_STEPS + 1):
    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)

    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    optimizer.zero_grad(set_to_none=True)

    output = model(**batch)
    loss = output.loss

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        max_norm=1.0,
    )

    optimizer.step()

    training_history.append({
        "step": step,
        "training_loss": loss.item(),
    })

    if step == 1 or step % 5 == 0 or step == TRAINING_STEPS:
        print(
            f"Step {step:3d}/{TRAINING_STEPS} "
            f"| Loss: {loss.item():.4f}"
        )

print("\nTraining complete.")

Step   1/30 | Loss: 6.1120
Step   5/30 | Loss: 4.8022
Step  10/30 | Loss: 4.7360
Step  15/30 | Loss: 4.8111
Step  20/30 | Loss: 4.2935
Step  25/30 | Loss: 4.1758
Step  30/30 | Loss: 4.2858

Training complete.


In [23]:
### Individual training losses may fluctuate because steps use different sentences and training enables dropout.

# Evaluate after training

In [24]:
after_eval_loss = evaluate_loss(eval_loader)

results_df = pd.DataFrame([
    {
        "stage": "Before BitFit",
        "evaluation_loss": before_eval_loss,
        "perplexity": math.exp(before_eval_loss),
    },
    {
        "stage": "After BitFit",
        "evaluation_loss": after_eval_loss,
        "perplexity": math.exp(after_eval_loss),
    },
])

display(results_df)

relative_change = (
    (before_eval_loss - after_eval_loss)
    / before_eval_loss
    * 100
)

print(f"Evaluation loss reduction: {relative_change:.2f}%")

if after_eval_loss < before_eval_loss:
    print("The model improved on this small held-out text set.")
else:
    print(
        "Held-out loss did not improve in this run. "
        "Bias-only training does not guarantee improvement."
    )

,stage,evaluation_loss,perplexity
0,Before BitFit,5.439722,230.378021
1,After BitFit,4.544838,94.145140


Evaluation loss reduction: 16.45%
The model improved on this small held-out text set.


# Compare text completions

In [25]:
after_completions = {
    prompt: complete_text(prompt)
    for prompt in test_prompts
}

for prompt in test_prompts:
    print("\nPROMPT:", prompt)

    print("\nBEFORE:")
    print(before_completions[prompt])

    print("\nAFTER:")
    print(after_completions[prompt])

    print("=" * 70)


PROMPT: Predictive maintenance is

BEFORE:
a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is

AFTER:
important for maintenance.

PROMPT: When bearing temperature increases,

BEFORE:
the temperature increases, and the temperature decreases.

AFTER:
the temperature of the body increases.

PROMPT: Vibration monitoring helps engineers

BEFORE:
to understand the impact of the stress on the brain.

AFTER:
monitor the condition of the system.


In [26]:
## The completions may still be repetitive or inaccurate. A lower held-out loss is useful evidence for this exercise, but it does not establish factual accuracy or broad domain expertise.

In [27]:
# Verify which parameters changed

In [28]:
after_fingerprints = parameter_fingerprints()

changed_biases = []
changed_frozen_parameters = []

for name, parameter in model.named_parameters():
    changed = (
        before_fingerprints[name] != after_fingerprints[name]
    )

    if changed:
        if parameter.requires_grad:
            changed_biases.append(name)
        else:
            changed_frozen_parameters.append(name)

print("Changed bias tensors:", len(changed_biases))
print("Changed frozen tensors:", len(changed_frozen_parameters))

assert len(changed_biases) > 0, "No bias parameters changed."
assert len(changed_frozen_parameters) == 0, (
    "A frozen parameter unexpectedly changed."
)

print("\nVerified: bias parameters changed; frozen parameters stayed identical.")

display(pd.DataFrame({
    "changed_bias_parameter": changed_biases
}))

Changed bias tensors: 37
Changed frozen tensors: 0

Verified: bias parameters changed; frozen parameters stayed identical.


,changed_bias_parameter
0,transformer.h.0.ln_1.bias
1,transformer.h.0.attn.c_attn.bias
2,transformer.h.0.attn.c_proj.bias
3,transformer.h.0.ln_2.bias
4,transformer.h.0.mlp.c_fc.bias
5,transformer.h.0.mlp.c_proj.bias
6,transformer.h.1.ln_1.bias
7,transformer.h.1.attn.c_attn.bias
8,transformer.h.1.attn.c_proj.bias
9,transformer.h.1.ln_2.bias


# ## Assignment

1. Explain the difference between a weight and a bias.

2. Why do we pass only trainable_parameters to the optimizer?

3. Does a reduction in training loss guarantee better held-out performance?

4. Repeat the experiment with 60 training steps. (from load tokenizer till last)

5. Repeat with LEARNING_RATE = 1e-4.
   Keep the data and training steps unchanged.

6. Compare held-out loss across runs.
   Are longer training runs always better?



## Main takeaway

BitFit adapts the model by updating existing bias parameters.

It does not add prompt vectors, prefix vectors, adapter modules,
or low-rank matrices.